# 🎬 GPU BRIDGE v2 — ek Run, phir so jao

1. `Runtime` → **Change runtime type** → **T4 GPU** → Save
2. **▶ Run** (sirf ek cell)
3. Tab khula chhod dein. **Bas.**

URL aur commands dono khud pahunch jayenge — kuch copy-paste nahi:

| Channel | Kaam |
|---|---|
| **ntfy.sh** (`monarch-chaos-gpu-742683`) | endpoint URL ping (agent khud padh leta hai) |
| **Drive `Chaos video 2/CHAOS_GPU_CMD.json`** | agent command likhta hai |
| **Drive `Chaos video 2/CHAOS_GPU_RESULT.json`** | result jahan likha jata hai |
| **Drive `Chaos video 2/CHAOS_GPU_HEARTBEAT.json`** | har 30 s: GPU, jobs, URL |
| **Drive `Chaos video 2/chaos_gpu_clips/`** | banayi hui files |

Tunnel mar jaye to bhi kaam chalta rahega — Drive channel alag hai.


In [ ]:

# ══════════════════════════════════════════════════════════════════════════
#  MONARCH · CHAOS — GPU BRIDGE v2      (ek Run, phir so jao)
# ══════════════════════════════════════════════════════════════════════════
import os, sys, json, time, base64, pathlib, subprocess, threading, secrets, re, shutil
from datetime import datetime

TOPIC = "monarch-chaos-gpu-742683"
DRIVE_FOLDER_NAME = "Chaos video 2"
PORT = 8000
CONTENT = pathlib.Path(os.environ.get("CHAOS_CONTENT", "/content"))
try:
    CONTENT.mkdir(parents=True, exist_ok=True)
except PermissionError:                      # local test / limited sandbox
    CONTENT = pathlib.Path(os.environ.get("TMPDIR", "/tmp")) / "chaos_content"
    CONTENT.mkdir(parents=True, exist_ok=True)
WORK = CONTENT / "chaos_work"; WORK.mkdir(exist_ok=True)
TOKEN = "monarch-chaos-742683"          # fixed on purpose: the agent already knows it
os.environ["CHAOS_TOKEN"] = TOKEN
os.environ["CHAOS_WORK"] = str(WORK)
os.environ["PORT"] = str(PORT)
os.environ["CHAOS_TAIL"] = "25"

def say(*a):
    print(datetime.now().strftime("%H:%M:%S"), *a, flush=True)

# ── 0 · GPU ──────────────────────────────────────────────────────────────
say("· GPU check")
try:
    smi = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                         capture_output=True, text=True, timeout=30).stdout.strip()
except Exception:
    smi = ""
say("  nvidia-smi:", smi or "NOTHING")
try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
    VRAM = round(torch.cuda.get_device_properties(0).total_memory/1024**3,1) if HAS_CUDA else 0.0
    say(f"  torch {torch.__version__} cuda={torch.version.cuda} available={HAS_CUDA} vram={VRAM}GB")
except Exception as e:
    HAS_CUDA, VRAM = False, 0.0
    say("  torch missing:", type(e).__name__, "(Colab par hota hai — yahan test me nahi)")
if not HAS_CUDA:
    say("  ⚠ GPU nahi mila — Runtime → Change runtime type → T4 GPU → phir Run")
    say("  (server phir bhi chalega: smoke/shell commands kaam karenge)")

# ── 1 · deps ─────────────────────────────────────────────────────────────
say("· deps")
for pkg in ("flask", "imageio-ffmpeg"):
    r = subprocess.run([sys.executable,"-m","pip","install","-q",pkg],
                       capture_output=True, text=True)
    if r.returncode != 0:                    # PEP-668 boxes
        subprocess.run([sys.executable,"-m","pip","install","-q",
                        "--break-system-packages",pkg], check=False,
                       capture_output=True)
say("  deps ready")

# ── 2 · Drive ────────────────────────────────────────────────────────────
say("· Drive")
drive_root = None
folder = None
try:
    from google.colab import drive as gdrive
    gdrive.mount("/content/drive", force_remount=False)
    drive_root = pathlib.Path("/content/drive/MyDrive")
    base_depth = len(str(drive_root).split(os.sep))
    for root, dirs, files in os.walk(drive_root):
        if pathlib.Path(root).name == DRIVE_FOLDER_NAME:
            folder = pathlib.Path(root); break
        if len(root.split(os.sep)) - base_depth > 3:
            dirs[:] = []
    folder = folder or drive_root
    (folder / "chaos_gpu_clips").mkdir(parents=True, exist_ok=True)
    say("  folder:", folder)
except Exception as e:
    say("  Drive mount FAILED:", type(e).__name__, e)
    folder = None

# ── 3 · server + bridge (embedded) ───────────────────────────────────────
say("· server source")
(CONTENT / "agent_server.py").write_bytes(base64.b64decode("..."))
(CONTENT / "drive_bridge.py").write_bytes(base64.b64decode("IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKTU9OQVJDSCDCtyBDSEFPUyDigJQgRHJpdmUgY29tbWFuZCBicmlkZ2UgKHJ1bnMgaW5zaWRlIENvbGFiLCBuZXh0IHRvIGFnZW50X3NlcnZlcikuCgpXaHkgdGhpcyBleGlzdHMKLS0tLS0tLS0tLS0tLS0tClR1bm5lbHMgZGllLCBVUkxzIGV4cGlyZSwgYW5kIHRoZSBhZ2VudCdzIHNoZWxsIGNhbm5vdCByZWFjaCB0aGUgb3BlbiBpbnRlcm5ldC4KRHJpdmUgZG9lcyBub3QgaGF2ZSBlaXRoZXIgcHJvYmxlbTogdGhlIGFnZW50IGNhbiBhbHJlYWR5IHdyaXRlIGEgZmlsZSBpbnRvIHRoZQpwcm9qZWN0J3MgRHJpdmUgZm9sZGVyIGFuZCByZWFkIG9uZSBiYWNrLiBTbyBEcml2ZSBpcyB0aGUgY29udHJvbCBjaGFubmVsOgoKICAgIERyaXZlOiBDSEFPU19HUFVfQ01ELmpzb24gICAgICA8LSB0aGUgYWdlbnQgd3JpdGVzIGEgY29tbWFuZCBoZXJlCiAgICBEcml2ZTogQ0hBT1NfR1BVX1JFU1VMVC5qc29uICAgPC0gd2Ugd3JpdGUgdGhlIGFuc3dlciBoZXJlCiAgICBEcml2ZTogQ0hBT1NfR1BVX0hFQVJUQkVBVC5qc29uPC0gZXZlcnkgMzAgczogYWxpdmUsIEdQVSwgam9icywgdHVubmVsIFVSTAogICAgRHJpdmU6IGNoYW9zX2dwdV9jbGlwcy8gICAgICAgIDwtIHByb2R1Y2VkIGZpbGVzIGdldCBjb3BpZWQgaGVyZQoKQSB0dW5uZWwgaXMgc3RpbGwgc3RhcnRlZCAoc28gdGhlIGFnZW50IGNhbiBhbHNvIGRyaXZlIHRoZSBIVFRQIEFQSSBkaXJlY3RseSB3aGVuCml0IGlzIHJlYWNoYWJsZSksIGFuZCB0aGUgVVJMIGlzIHB1Ymxpc2hlZCB0byBudGZ5LnNoIHNvIGl0IGNvc3RzIHplcm8gY29weS1wYXN0ZS4KQnV0IG5vdGhpbmcgZGVwZW5kcyBvbiB0aGUgdHVubmVsOiBpZiBpdCBpcyBkb3duLCB3b3JrIHN0aWxsIGZsb3dzIHRocm91Z2ggRHJpdmUuCgpQcm90b2NvbCAob25lIEpTT04gb2JqZWN0IHBlciBmaWxlLCBhdG9taWMgd3JpdGVzIHZpYSAudG1wICsgcmVwbGFjZSkKICAgIHsiaWQiOiAiYzAwMSIsICJhY3Rpb24iOiAicnVuIiwgICAicmVjaXBlIjogInNtb2tlIiwgInBhcmFtcyI6IHsuLi59fQogICAgeyJpZCI6ICJjMDAyIiwgImFjdGlvbiI6ICJzaGVsbCIsICJjbWQiOiAicGlwIGluc3RhbGwgLi4uIiwgIm1vZGUiOiAic3luYyJ9CiAgICB7ImlkIjogImMwMDMiLCAiYWN0aW9uIjogImpvYiIsICAgImpvYiI6ICJhYmMxMjMifQogICAgeyJpZCI6ICJjMDA0IiwgImFjdGlvbiI6ICJzdGF0dXMifQogICAgeyJpZCI6ICJjMDA1IiwgImFjdGlvbiI6ICJmaWxlcyJ9CiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHNodXRpbAppbXBvcnQgdGltZQppbXBvcnQgdXJsbGliLnBhcnNlCmltcG9ydCB1cmxsaWIucmVxdWVzdApmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZXpvbmUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpDTURfTkFNRSA9ICJDSEFPU19HUFVfQ01ELmpzb24iClJFU19OQU1FID0gIkNIQU9TX0dQVV9SRVNVTFQuanNvbiIKQkVBVF9OQU1FID0gIkNIQU9TX0dQVV9IRUFSVEJFQVQuanNvbiIKQ0xJUFMgPSAiY2hhb3NfZ3B1X2NsaXBzIgoKCmRlZiBub3coKSAtPiBzdHI6CiAgICByZXR1cm4gZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0Yykuc3RyZnRpbWUoIiVZLSVtLSVkVCVIOiVNOiVTWiIpCgoKZGVmIHdyaXRlX2pzb24ocGF0aDogUGF0aCwgb2JqOiBkaWN0KSAtPiBOb25lOgogICAgdG1wID0gcGF0aC53aXRoX3N1ZmZpeChwYXRoLnN1ZmZpeCArICIudG1wIikKICAgIHRtcC53cml0ZV90ZXh0KGpzb24uZHVtcHMob2JqLCBpbmRlbnQ9MSkpCiAgICB0bXAucmVwbGFjZShwYXRoKQoKCmRlZiByZWFkX2pzb24ocGF0aDogUGF0aCk6CiAgICB0cnk6CiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoKSkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIE5vbmUKCgpjbGFzcyBCcmlkZ2U6CiAgICBkZWYgX19pbml0X18oc2VsZiwgZm9sZGVyOiBQYXRoLCB0b2tlbjogc3RyLCB1cmxfZ2V0dGVyPU5vbmUsIHBvcnQ6IGludCA9IDgwMDApOgogICAgICAgIHNlbGYuZm9sZGVyID0gZm9sZGVyCiAgICAgICAgc2VsZi50b2tlbiA9IHRva2VuCiAgICAgICAgc2VsZi51cmxfZ2V0dGVyID0gdXJsX2dldHRlciBvciAobGFtYmRhOiBOb25lKQogICAgICAgIHNlbGYucG9ydCA9IHBvcnQKICAgICAgICBzZWxmLmxhc3RfaWQgPSBOb25lCiAgICAgICAgc2VsZi5wcm9jZXNzZWQgPSAwCiAgICAgICAgc2VsZi5jbGlwcyA9IGZvbGRlciAvIENMSVBTCiAgICAgICAgc2VsZi5jbGlwcy5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGh0dHAgLS0tCiAgICBkZWYgYXBpKHNlbGYsIHBhdGg6IHN0ciwgKipwYXJhbXMpIC0+IGRpY3Q6CiAgICAgICAgcGFyYW1zWyJ0b2tlbiJdID0gc2VsZi50b2tlbgogICAgICAgIHEgPSB1cmxsaWIucGFyc2UudXJsZW5jb2RlKHBhcmFtcykKICAgICAgICB1cmwgPSBmImh0dHA6Ly8xMjcuMC4wLjE6e3NlbGYucG9ydH17cGF0aH0/e3F9IgogICAgICAgIHRyeToKICAgICAgICAgICAgd2l0aCB1cmxsaWIucmVxdWVzdC51cmxvcGVuKHVybCwgdGltZW91dD02MCkgYXMgcjoKICAgICAgICAgICAgICAgIHJldHVybiBqc29uLmxvYWRzKHIucmVhZCgpLmRlY29kZSgpKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgcmV0dXJuIHsib2siOiBGYWxzZSwgImVycm9yIjogZiJ7dHlwZShlKS5fX25hbWVfX306IHtlfSJ9CgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gZmlsZXMgLS0tCiAgICBkZWYgc3luY19jbGlwcyhzZWxmKSAtPiBsaXN0W3N0cl06CiAgICAgICAgbW92ZWQgPSBbXQogICAgICAgIG91dCA9IFBhdGgob3MuZW52aXJvbi5nZXQoIkNIQU9TX1dPUksiLCAiL2NvbnRlbnQvY2hhb3Nfd29yayIpKSAvICJvdXQiCiAgICAgICAgaWYgbm90IG91dC5leGlzdHMoKToKICAgICAgICAgICAgcmV0dXJuIG1vdmVkCiAgICAgICAgZm9yIGYgaW4gb3V0Lmdsb2IoIioiKToKICAgICAgICAgICAgaWYgbm90IGYuaXNfZmlsZSgpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZHN0ID0gc2VsZi5jbGlwcyAvIGYubmFtZQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBpZiBub3QgZHN0LmV4aXN0cygpIG9yIGRzdC5zdGF0KCkuc3Rfc2l6ZSAhPSBmLnN0YXQoKS5zdF9zaXplOgogICAgICAgICAgICAgICAgICAgIHNodXRpbC5jb3B5MihmLCBkc3QpCiAgICAgICAgICAgICAgICAgICAgbW92ZWQuYXBwZW5kKGYubmFtZSkKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgIHBhc3MKICAgICAgICByZXR1cm4gbW92ZWQKCiAgICBkZWYgZmlsZV9saXN0KHNlbGYpIC0+IGxpc3RbZGljdF06CiAgICAgICAgcm93cyA9IFtdCiAgICAgICAgb3V0ID0gUGF0aChvcy5lbnZpcm9uLmdldCgiQ0hBT1NfV09SSyIsICIvY29udGVudC9jaGFvc193b3JrIikpIC8gIm91dCIKICAgICAgICBmb3IgZiBpbiBzb3J0ZWQob3V0Lmdsb2IoIioiKSkgaWYgb3V0LmV4aXN0cygpIGVsc2UgW106CiAgICAgICAgICAgIGlmIGYuaXNfZmlsZSgpOgogICAgICAgICAgICAgICAgcm93cy5hcHBlbmQoeyJuYW1lIjogZi5uYW1lLCAiYnl0ZXMiOiBmLnN0YXQoKS5zdF9zaXplfSkKICAgICAgICByZXR1cm4gcm93cwoKICAgIGRlZiB3YWl0X2pvYihzZWxmLCBqaWQ6IHN0ciwgdGltZW91dDogZmxvYXQpIC0+IGRpY3Q6CiAgICAgICAgIiIiUG9sbCBhIGpvYiB1bnRpbCBpdCBmaW5pc2hlcyBzbyB0aGUgUkVTVUxUIGZpbGUgY2FycmllcyB0aGUgb3V0Y29tZS4iIiIKICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgbGFzdCA9IHt9CiAgICAgICAgd2hpbGUgdGltZS50aW1lKCkgLSB0MCA8IHRpbWVvdXQ6CiAgICAgICAgICAgIGxhc3QgPSBzZWxmLmFwaSgiL2pvYiIsIGlkPWppZCwgdGFpbD1pbnQob3MuZW52aXJvbi5nZXQoIkNIQU9TX1RBSUwiLCAiMjUiKSkpCiAgICAgICAgICAgIHN0ID0gKGxhc3QuZ2V0KCJqb2IiKSBvciB7fSkuZ2V0KCJzdGF0ZSIpCiAgICAgICAgICAgIGlmIHN0IGluICgiZG9uZSIsICJmYWlsZWQiLCAiY2FuY2VsbGVkIik6CiAgICAgICAgICAgICAgICByZXR1cm4gbGFzdAogICAgICAgICAgICB0aW1lLnNsZWVwKDIuMCkKICAgICAgICBsYXN0WyJvayJdID0gRmFsc2UKICAgICAgICBsYXN0WyJlcnJvciJdID0gZiJzdGlsbCBydW5uaW5nIGFmdGVyIHt0aW1lb3V0Oi4wZn1zIOKAlCBwb2xsIC9qb2Igb3IgcmVhZCBSRVNVTFQgYWdhaW4iCiAgICAgICAgcmV0dXJuIGxhc3QKCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBjb21tYW5kcyAtLS0KICAgIGRlZiBoYW5kbGUoc2VsZiwgY21kOiBkaWN0KSAtPiBkaWN0OgogICAgICAgIGFjdCA9IGNtZC5nZXQoImFjdGlvbiIsICJzdGF0dXMiKQogICAgICAgIGlmIGFjdCA9PSAic3RhdHVzIjoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuYXBpKCIvc3RhdHVzIikKICAgICAgICBpZiBhY3QgPT0gInJlY2lwZXMiOgogICAgICAgICAgICByZXR1cm4gc2VsZi5hcGkoIi9yZWNpcGVzIikKICAgICAgICBpZiBhY3QgPT0gImZpbGVzIjoKICAgICAgICAgICAgcmV0dXJuIHsib2siOiBUcnVlLCAiZmlsZXMiOiBzZWxmLmZpbGVfbGlzdCgpLCAiY2xpcHNfZGlyIjogc3RyKHNlbGYuY2xpcHMpfQogICAgICAgIGlmIGFjdCA9PSAiam9iIjoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuYXBpKCIvam9iIiwgaWQ9Y21kLmdldCgiam9iIiwgIiIpLCB0YWlsPWNtZC5nZXQoInRhaWwiLCA0MCkpCiAgICAgICAgaWYgYWN0ID09ICJsb2ciOgogICAgICAgICAgICByZXR1cm4gc2VsZi5hcGkoIi9sb2ciLCBpZD1jbWQuZ2V0KCJqb2IiLCAiIiksIG9mZnNldD1jbWQuZ2V0KCJvZmZzZXQiLCAwKSkKICAgICAgICBpZiBhY3QgPT0gImNhbmNlbCI6CiAgICAgICAgICAgIHJldHVybiBzZWxmLmFwaSgiL2NhbmNlbCIsIGlkPWNtZC5nZXQoImpvYiIsICIiKSkKICAgICAgICBpZiBhY3QgaW4gKCJydW4iLCAic2hlbGwiKToKICAgICAgICAgICAgcGFyYW1zID0geyJyZWNpcGUiOiBjbWQuZ2V0KCJyZWNpcGUiLCAic21va2UiKX0KICAgICAgICAgICAgaWYgYWN0ID09ICJzaGVsbCI6CiAgICAgICAgICAgICAgICBwYXJhbXMgPSB7InJlY2lwZSI6ICJzaGVsbCIsICJjbWQiOiBjbWQuZ2V0KCJjbWQiLCAiIiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgIm1vZGUiOiBjbWQuZ2V0KCJtb2RlIiwgInN5bmMiKX0KICAgICAgICAgICAgcGFyYW1zLnVwZGF0ZSh7azogc3RyKHYpIGZvciBrLCB2IGluIGNtZC5nZXQoInBhcmFtcyIsIHt9KS5pdGVtcygpfSkKICAgICAgICAgICAgc3RhcnRlZCA9IHNlbGYuYXBpKCIvcnVuIiwgKipwYXJhbXMpCiAgICAgICAgICAgIGppZCA9IHN0YXJ0ZWQuZ2V0KCJqb2IiKQogICAgICAgICAgICB3YWl0ID0gZmxvYXQoY21kLmdldCgid2FpdCIsIDMwMCkpCiAgICAgICAgICAgIGlmIGppZCBhbmQgd2FpdCA+IDA6CiAgICAgICAgICAgICAgICBkb25lID0gc2VsZi53YWl0X2pvYihqaWQsIHdhaXQpCiAgICAgICAgICAgICAgICBkb25lWyJzdGFydGVkIl0gPSBzdGFydGVkCiAgICAgICAgICAgICAgICByZXR1cm4gZG9uZQogICAgICAgICAgICByZXR1cm4gc3RhcnRlZAogICAgICAgIHJldHVybiB7Im9rIjogRmFsc2UsICJlcnJvciI6IGYidW5rbm93biBhY3Rpb24ge2FjdH0ifQoKICAgICMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gcnVuIC0tLQogICAgZGVmIG9uY2Uoc2VsZikgLT4gYm9vbDoKICAgICAgICBwID0gc2VsZi5mb2xkZXIgLyBDTURfTkFNRQogICAgICAgIGNtZCA9IHJlYWRfanNvbihwKQogICAgICAgIGlmIG5vdCBjbWQgb3Igbm90IGNtZC5nZXQoImlkIikgb3IgY21kWyJpZCJdID09IHNlbGYubGFzdF9pZDoKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgY2lkID0gc3RyKGNtZFsiaWQiXSkKICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgcmVzID0gc2VsZi5oYW5kbGUoY21kKQogICAgICAgIHNlbGYubGFzdF9pZCA9IGNpZAogICAgICAgIHNlbGYucHJvY2Vzc2VkICs9IDEKICAgICAgICBtb3ZlZCA9IHNlbGYuc3luY19jbGlwcygpCiAgICAgICAgcGF5bG9hZCA9IHsiY21kX2lkIjogY2lkLCAiYWN0aW9uIjogY21kLmdldCgiYWN0aW9uIiksICJhdCI6IG5vdygpLAogICAgICAgICAgICAgICAgICAgInNlY29uZHMiOiByb3VuZCh0aW1lLnRpbWUoKSAtIHQwLCAxKSwgIm9rIjogYm9vbChyZXMuZ2V0KCJvayIsIFRydWUpKSwKICAgICAgICAgICAgICAgICAgICJyZXN1bHQiOiByZXMsICJmaWxlcyI6IHNlbGYuZmlsZV9saXN0KCksICJzeW5jZWQiOiBtb3ZlZCwKICAgICAgICAgICAgICAgICAgICJwcm9jZXNzZWQiOiBzZWxmLnByb2Nlc3NlZH0KICAgICAgICB3cml0ZV9qc29uKHNlbGYuZm9sZGVyIC8gUkVTX05BTUUsIHBheWxvYWQpCiAgICAgICAgcHJpbnQoZiJbYnJpZGdlXSB7Y2lkfSB7Y21kLmdldCgnYWN0aW9uJyl9IG9rPXtwYXlsb2FkWydvayddfSAiCiAgICAgICAgICAgICAgZiJpbiB7cGF5bG9hZFsnc2Vjb25kcyddfXMiLCBmbHVzaD1UcnVlKQogICAgICAgIHJldHVybiBUcnVlCgogICAgZGVmIGhlYXJ0YmVhdChzZWxmKSAtPiBOb25lOgogICAgICAgIHN0ID0gc2VsZi5hcGkoIi9zdGF0dXMiKQogICAgICAgIGJlYXQgPSB7ImFsaXZlIjogVHJ1ZSwgImF0Ijogbm93KCksICJ1cmwiOiBzZWxmLnVybF9nZXR0ZXIoKSwKICAgICAgICAgICAgICAgICJwcm9jZXNzZWQiOiBzZWxmLnByb2Nlc3NlZCwgImxhc3RfY21kIjogc2VsZi5sYXN0X2lkLAogICAgICAgICAgICAgICAgImdwdSI6IHN0LmdldCgiZ3B1Iiwge30pLCAiam9icyI6IHN0LmdldCgiam9iX3N0YXRlcyIsIHt9KSwKICAgICAgICAgICAgICAgICJxdWV1ZSI6IHN0LmdldCgicXVldWUiKSwgImZpbGVzIjogc2VsZi5maWxlX2xpc3QoKX0KICAgICAgICB3cml0ZV9qc29uKHNlbGYuZm9sZGVyIC8gQkVBVF9OQU1FLCBiZWF0KQoKICAgIGRlZiBsb29wKHNlbGYsIGludGVydmFsOiBmbG9hdCA9IDUuMCkgLT4gTm9uZToKICAgICAgICBsYXN0X2JlYXQgPSAwLjAKICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBzZWxmLm9uY2UoKQogICAgICAgICAgICAgICAgaWYgdGltZS50aW1lKCkgLSBsYXN0X2JlYXQgPiAzMDoKICAgICAgICAgICAgICAgICAgICBzZWxmLmhlYXJ0YmVhdCgpCiAgICAgICAgICAgICAgICAgICAgbGFzdF9iZWF0ID0gdGltZS50aW1lKCkKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICAgICAgcHJpbnQoZiJbYnJpZGdlXSBsb29wIGVycm9yIHt0eXBlKGUpLl9fbmFtZV9ffToge2V9IiwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgdGltZS5zbGVlcChpbnRlcnZhbCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgZm9sZGVyID0gUGF0aChvcy5lbnZpcm9uLmdldCgiQ0hBT1NfRFJJVkVfRElSIiwgIi4iKSkKICAgIHRva2VuID0gb3MuZW52aXJvbi5nZXQoIkNIQU9TX1RPS0VOIiwgIiIpCiAgICBCcmlkZ2UoZm9sZGVyLCB0b2tlbikubG9vcCgpCg=="))
sys.path.insert(0, str(CONTENT))

import agent_server as SRV
SRV.TOKEN = TOKEN
SRV.WORK = WORK
SRV.OUT = WORK / "out"; SRV.OUT.mkdir(parents=True, exist_ok=True)
SRV.LOGS = WORK / "logs"; SRV.LOGS.mkdir(parents=True, exist_ok=True)

SRV_URL = {"url": None}
def run_server():
    while True:
        try:
            SRV.serve(port=PORT)
        except Exception as e:
            say("  server crashed:", type(e).__name__, e)
        time.sleep(3)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)
say("  server thread up")

# ── 4 · tunnel ───────────────────────────────────────────────────────────
def start_cloudflared():
    exe = CONTENT / "cloudflared"
    if not exe.exists():
        url = ("https://github.com/cloudflare/cloudflared/releases/latest/download/"
               "cloudflared-linux-amd64")
        r = subprocess.run(["curl","-fsSL","-o",str(exe),url], capture_output=True, text=True)
        if r.returncode != 0:
            return None, "download: " + (r.stderr or "")[-160:]
        exe.chmod(0o755)
    log = CONTENT / "cf.log"
    log.write_text("")
    subprocess.Popen([str(exe),"tunnel","--url",f"http://127.0.0.1:{PORT}",
                      "--no-autoupdate","--logfile",str(log)],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(75):
        time.sleep(1)
        txt = log.read_text(errors="ignore") if log.exists() else ""
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
        if m:
            return m.group(0), None
    return None, "timeout"

def start_lhr():
    p = subprocess.Popen(["ssh","-o","StrictHostKeyChecking=no","-o","ServerAliveInterval=30",
                          "-R",f"80:127.0.0.1:{PORT}","nokey@localhost.run"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(60):
        line = p.stdout.readline()
        if not line:
            time.sleep(1); continue
        m = re.search(r"https://[a-z0-9-]+\.lhr\.life", line)
        if m:
            return m.group(0), None
    return None, "no url"

def open_tunnel():
    say("· tunnel")
    u, e1 = start_cloudflared()
    if not u:
        say("  cloudflared:", e1)
        u, e2 = start_lhr()
        if not u:
            say("  localhost.run:", e2)
    if u:
        say("  URL:", u)
    else:
        say("  ⚠ koi tunnel nahi — Drive channel phir bhi chalega")
    SRV_URL["url"] = u
    return u

# ── 5 · publish (ntfy + Drive) ───────────────────────────────────────────
def publish(url):
    if url:
        try:
            import urllib.request
            urllib.request.urlopen(urllib.request.Request(
                f"https://ntfy.sh/{TOPIC}", data=url.encode()), timeout=25)
            say("  ntfy ping ok")
        except Exception as e:
            say("  ntfy ping failed:", type(e).__name__)
    payload = {"url": url, "token": TOKEN, "port": PORT, "vram_gb": VRAM,
               "created": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
               "ntfy": f"https://ntfy.sh/{TOPIC}/json?poll=1",
               "how": "GET <url>/status?token=<token>  |  or write CHAOS_GPU_CMD.json"}
    targets = []
    if folder:
        targets.append(folder)
    targets.append(pathlib.Path("/content/drive/MyDrive"))
    targets.append(CONTENT)
    for t in targets:
        try:
            t.mkdir(parents=True, exist_ok=True)
            (t / "CHAOS_GPU_ENDPOINT.json").write_text(json.dumps(payload, indent=1))
            say("  endpoint written:", t)
        except Exception as e:
            say("  endpoint write skipped:", t, type(e).__name__)

url = open_tunnel()
publish(url)

# ── 6 · Drive command bridge ─────────────────────────────────────────────
import drive_bridge as DB
if folder:
    bridge = DB.Bridge(folder, TOKEN, url_getter=lambda: SRV_URL["url"], port=PORT)
    threading.Thread(target=bridge.loop, kwargs={"interval": 5.0}, daemon=True).start()
    say("  bridge running (poll 5 s) — commands: CHAOS_GPU_CMD.json")
else:
    bridge = None
    say("  ⚠ bridge OFF (no Drive) — sirf tunnel se control hoga")

say("")
say("═" * 66)
say(" READY — kuch copy karne ki zaroorat nahi. Tab khula chhod dein.")
say(" endpoint:", url or "none")
say(" token:", TOKEN)
if not HAS_CUDA:
    say(" ⚠⚠ GPU OFF: Runtime → Change runtime type → T4 GPU → phir yeh cell dobara Run")
say("═" * 66)

# ── 7 · supervisor (yahi cell chalta rehta hai) ──────────────────────────
tick = 0
while True:
    time.sleep(60)
    tick += 1
    try:
        import urllib.request
        alive = False
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=8) as r:
                alive = b"ok" in r.read()
        except Exception:
            alive = False
        if not alive:
            say("supervisor: server dead → restart")
            threading.Thread(target=run_server, daemon=True).start()
            time.sleep(4)
        if tick % 10 == 0:                    # every 10 min: refresh the tunnel URL
            new = open_tunnel()
            if new:
                publish(new)
        if tick % 5 == 0:
            say("alive | url:", (SRV_URL["url"] or "none")[-32:],
                "| processed:", getattr(bridge, "processed", 0))
    except Exception as e:
        say("supervisor error:", type(e).__name__, e)
